# k-space — mejorar Normal vs anormal

Continuación de `eda_kspace.ipynb`. Allí `log|K|` daba AUC Normal-vs-anormal ≈ 0.84 con `PCA(64) + LogisticRegression` entrenado a 4 clases.
Aquí el objetivo es **solo binario** (Normal = 0; FC, W y WC = anormal) y **todas las mejoras se hacen en el dominio k-space**.

**Ideas a probar**

| | Idea | Por qué podría ayudar |
|---|---|---|
| H1 | Entrenar directamente el binario y cambiar el clasificador (sin PCA, LDA con *shrinkage*) | `PCA(64)` descarta varianza pequeña que quizá es justo la del crepitante |
| H2 | **Pooling log-espaciado** de `log|K|` en `ku` y `kv` (*modulation spectrum* en escala log) | 3 000 píxeles ruidosos → ~144 celdas con más sentido físico, menos sobreajuste |
| H3 | **k-space local**: FFT 2D por parches de 16×16 (0.5 s) y resumen `mean`/`max` entre parches | un crepitante aislado en 3 s se diluye en la FFT global; por parche aparece y el `max` lo conserva |
| H4 | Mantener la posición en frecuencia (pooling solo sobre el tiempo) | wheeze y crackle viven en bandas de frecuencia distintas |
| H5 | Combinar bloques (global + local) con pesos iguales por bloque | información complementaria |
| H6 | **CNN** sobre el espectrograma, sobre la imagen de k-space (log-magnitud) y fusionando ambas | aprender los filtros en vez de fijarlos a mano (parches, pooling) |

**Protocolo (2025 es solo la prueba).**
- **Desarrollo** (`AUC dev`): *leave-one-train-year-out* dentro de 2022–2024: se entrena con 2 de esos años y se evalúa el tercero (3 folds). 2025 no entra en ningún entrenamiento ni en ninguna elección.
- **Prueba** (`AUC 2025`): entrenar con 2022–2024 y evaluar en 2025, una sola vez por modelo. Es el split oficial del dataset.
- Todo se **ordena y elige por `AUC dev`**; `AUC 2025` se reporta al lado y no se usa para decidir.
- Además: AUC contra cada subclase (en 2025), robustez a un canal sintético y *domain accuracy* (qué tan bien se adivina el año 2022/23/24: menor = más invariante, azar = 0.33).

Las características son sin estado (no ven etiquetas), así que se calculan una vez y la validación solo reparte filas.

**Ojo.** Aunque 2025 no se use para elegir, esta tabla lo muestra para muchos modelos: la cifra honesta es la de *un* modelo elegido por `AUC dev` y evaluado en 2025, sin volver a iterar contra él.

## 1. Setup y datos

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from sklearn.metrics import roc_curve, roc_auc_score

import kspace_lib as kl

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.0)
plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 200, "axes.titlesize": 12, "axes.titleweight": "bold"})

RANDOM_STATE = 42
PROC_ROOT = Path("../data/processed")
MAX_PER_YEAR_CLASS = 1200       # tope por (anio, clase); None = todo (mas lento)
MASK_POLICY = "inpaint"         # SpecAugment residual en train (ver eda_kspace.ipynb, seccion 2.1)
CHANNEL_SIGMA = 0.5             # canal sintetico para la prueba de robustez

# CNN (seccion 6). WANDB_PROJECT = None desactiva el logging; solo se suben metricas (ni audio ni espectrogramas)
WANDB_PROJECT = "biocas"        # mismo proyecto que workflow/train.py
WANDB_GROUP = "kspace-binary-cnn-v2"   # v2 = train 2022-24 -> test 2025 (v1 era LOYO con 2025 en el entrenamiento)
CNN_SEEDS = (0, 1)              # el AUC de cada fold se promedia entre semillas
CNN_EPOCHS = 30

In [ ]:
d = kl.load_data(PROC_ROOT, MAX_PER_YEAR_CLASS, RANDOM_STATE, MASK_POLICY)
X_chan = kl.channel_perturb(d.X, CHANNEL_SIGMA)
print("X:", d.X.shape)
pd.crosstab(d.year, pd.Series(d.y).map(dict(enumerate(kl.CLASSES))).values)[kl.CLASSES].assign(
    anormal=lambda t: t[["FC", "W", "WC"]].sum(1))

### 1.1 Características (se calculan una vez)

- `espectrograma`: píxeles del log-mel (referencia).
- `logK`: `log|K|` global sin `ku=0` (lo del notebook anterior).
- `logpool`: `log|K|` promediado en celdas log-espaciadas (9 bandas de `ku` × 16 de `kv` con signo = 144).
- `local16`: k-space por parches 16×16, paso 8 (7 posiciones en frecuencia × 10 en tiempo), con `mean` y `max` sobre parches.
  `_t` conserva la posición en frecuencia (7×16×8 = 896 por estadístico); `_all` colapsa todo (16×8 = 128).
- `local16x32`: igual con parches de 16 mel × 32 frames (1 s): más resolución en `ku`.

In [ ]:
def build(X):
    f = {"espectrograma": X.reshape(len(X), -1),
         "logK": kl.feat_logk(X),
         "logpool": kl.feat_logpool(X)}
    for tag, kw in (("local16", dict(pf=16, pt=16, sf=8, st=8)), ("local16x32", dict(pf=16, pt=32, sf=8, st=16))):
        loc = kl.feat_localk(X, **kw)
        f[f"{tag} mean_all"] = loc["mean_all"]
        f[f"{tag} mean+max_all"] = np.hstack([loc["mean_all"], loc["max_all"]])
        f[f"{tag} mean_t"] = loc["mean_t"]
        f[f"{tag} mean+max_t"] = np.hstack([loc["mean_t"], loc["max_t"]])
    return f


FEATS = build(d.X)
FEATS_CHAN = build(X_chan)
pd.Series({k: v.shape[1] for k, v in FEATS.items()}, name="dim").to_frame().T

## 2. Infraestructura de comparación

In [ ]:
DOM_IDX = kl.domain_indices(d)
DOM_CACHE = {}
ROWS = []


def summarize(r):
    # r: filas de kl.eval_binary / kc.eval_cnn. Se elige por "AUC dev" (2022-24); "AUC 2025" es solo reporte
    cl, ch = r[r.cond == "limpio"], r[r.cond == "canal"]
    te = cl[cl.split == "test"].iloc[0]
    return {"AUC dev": cl[cl.split == "dev"].auc.mean(), "AUC dev canal": ch[ch.split == "dev"].auc.mean(),
            "AUC 2025": te.auc, "AUC 2025 canal": ch[ch.split == "test"].auc.iloc[0], "bal_acc 2025": te.bal_acc,
            "AUC N-FC (2025)": te.auc_FC, "AUC N-W (2025)": te.auc_W, "AUC N-WC (2025)": te.auc_WC}


def bench(feat, clf_name, make_clf, F=None, F_chan=None, with_domain=True):
    # Evalua una (caracteristica, clasificador). `feat` es la clave en FEATS o un nombre libre si se pasa F/F_chan
    F = FEATS[feat] if F is None else F
    F_chan = FEATS_CHAN[feat] if F_chan is None else F_chan
    r = kl.eval_binary(d, F, F_chan, make_clf)
    key = feat.split(" [")[0]                  # el dominio no depende del clasificador ni de C
    if with_domain and key not in DOM_CACHE:
        DOM_CACHE[key] = kl.eval_domain(d, F, DOM_IDX)
    row = {"caracteristica": feat, "clasificador": clf_name, "dim": F.shape[1], **summarize(r), "domain acc": DOM_CACHE.get(key, np.nan)}
    ROWS.append(row)
    return row


def board(rows=None, sort="AUC dev"):
    t = pd.DataFrame(ROWS if rows is None else rows)
    return t.sort_values(sort, ascending=False).reset_index(drop=True).round(3)

## 3. H1 — clasificador sobre `log|K|` (binario)

Misma característica (`logK`, 3 008 números), distinto modelo. Se entrena directamente `y = anormal`.

In [ ]:
CLFS = {
    "LR+PCA64 (C=1)":     lambda: kl.clf_lr_pca(64, 1.0),
    "LR+PCA128 (C=1)":    lambda: kl.clf_lr_pca(128, 1.0),
    "LDA shrink+PCA128":  lambda: kl.clf_lda(128),
    "LR (C=1e-3)":        lambda: kl.clf_lr(1e-3),
    "LR (C=1e-4)":        lambda: kl.clf_lr(1e-4),
}
for nm, mk in tqdm(CLFS.items(), desc="clasificadores"):
    bench("logK", nm, mk)
board()

In [ ]:
sub = board()
best_lin_name = sub.loc[sub["AUC dev"].idxmax(), "clasificador"]
BEST_LIN = CLFS[best_lin_name]
print("clasificador lineal elegido (por AUC dev) para el resto:", best_lin_name)

per = []
for nm, mk in CLFS.items():
    r = kl.eval_binary(d, FEATS["logK"], FEATS_CHAN["logK"], mk)
    per.append(r[r.cond == "limpio"].set_index("hold").auc.rename(nm))
print("AUC por año evaluado (2022-24 = desarrollo con 2 años de entrenamiento; 2025 = prueba):")
pd.concat(per, axis=1).T.round(3)

## 4. H2–H4 — representaciones en k-space

Mismo clasificador lineal (el elegido arriba) sobre cada característica. Después un `HistGradientBoosting` solo sobre las compactas (≤ 400 números),
para ver si hay estructura no lineal que el modelo lineal no captura.

In [ ]:
ROWS_FEAT_START = len(ROWS)
for feat in tqdm(["espectrograma", "logpool", "local16 mean_all", "local16 mean+max_all", "local16 mean_t", "local16 mean+max_t",
                  "local16x32 mean_all", "local16x32 mean+max_all", "local16x32 mean_t", "local16x32 mean+max_t"], desc="caracteristicas"):
    bench(feat, f"{best_lin_name}", BEST_LIN)
board(ROWS[ROWS_FEAT_START:])

In [ ]:
for feat in tqdm(["logpool", "local16 mean_all", "local16 mean+max_all", "local16x32 mean+max_all"], desc="HGB"):
    bench(feat, "HistGradientBoosting", kl.clf_hgb)
board(rows=[r for r in ROWS if r["clasificador"] == "HistGradientBoosting"])

## 5. H5 — combinar bloques

Se concatenan bloques k-space y cada uno se divide por `sqrt(dim)` tras estandarizar, para que el bloque grande no ahogue a los pequeños.

In [ ]:
COMBOS = {
    "logpool + local16 mean+max_all":           ["logpool", "local16 mean+max_all"],
    "logpool + local16 mean+max_t":             ["logpool", "local16 mean+max_t"],
    "logpool + local16 mean+max_all + local16x32 mean+max_all": ["logpool", "local16 mean+max_all", "local16x32 mean+max_all"],
    "logK + local16 mean+max_t":                ["logK", "local16 mean+max_t"],
}
for nm, parts in tqdm(COMBOS.items(), desc="combos"):
    F = np.hstack([FEATS[p] for p in parts]); Fc = np.hstack([FEATS_CHAN[p] for p in parts])
    dims = [FEATS[p].shape[1] for p in parts]
    for C in (1e-2, 1e-1):
        bench(nm + f" [C={C:g}]", "LR+BlockScaler", (lambda C=C, dims=dims: kl.clf_lr(C, dims)), F=F, F_chan=Fc)
    bench(nm + " [HGB]", "HistGradientBoosting", kl.clf_hgb, F=F, F_chan=Fc) if F.shape[1] <= 600 else None
board(rows=[r for r in ROWS if "+" in r["caracteristica"]])

## 6. H6 — CNNs

Cuatro CNN pequeñas (4 bloques conv 3×3 con BatchNorm, 16-32-64-96 canales, *pooling* medio+máximo global, ~0.2 M parámetros por rama), todas con el mismo protocolo que los modelos lineales (3 folds de desarrollo en 2022–24 + prueba en 2025):

| Modelo | Entrada |
|---|---|
| `CNN espectrograma` | log-mel (64×94). Referencia |
| `CNN espectrograma − media temporal` | log-mel sin la media temporal por banda (= el notch `ku=0` en el dominio imagen) |
| `CNN k-space (log\|K\|)` | **imagen de k-space** (64×46, `kv` centrado, sin `ku=0`) |
| `CNN fusión espectrograma + k-space` | dos ramas, una por entrada, concatenadas antes de la capa final |

- Pérdida BCE con `pos_weight` (equivale al `class_weight="balanced"` de los lineales), AdamW + OneCycle, 30 épocas, parada temprana.
- La época se elige con un 10 % de las **grabaciones** del entrenamiento; el año evaluado nunca se mira y su AUC se registra una sola vez al final.
- Augmentation (solo en las ramas de espectrograma): desplazamiento circular en tiempo y canal suave aleatorio (σ = 0.3). `|K|` ya es invariante a ambos, así que la rama k-space no se aumenta: la pregunta es si la CNN sobre espectrograma *aprende* lo que k-space da gratis.
- Con `WANDB_PROJECT` se crea una run por (modelo, fold, semilla) con curvas por época (`dev2022`, `dev2023`, `dev2024`, `test2025`) y una run `…|resumen` por modelo.
- La AUC de validación interna sale muy alta (≈ 0.98 con espectrograma) porque comparte pacientes y año con el entrenamiento: solo sirve para elegir la época, no es una estimación del rendimiento.

In [ ]:
import kspace_cnn as kc

INP, INP_CHAN = kc.make_inputs(d.X), kc.make_inputs(X_chan)
print("dispositivo:", kc.DEV, "| entradas:", {k: v.shape[1:] for k, v in INP.items()})

CNN_MODELS = {
    "CNN espectrograma":                     ["spec"],
    "CNN espectrograma − media temporal":    ["spec_demean"],
    "CNN k-space (log|K|)":                  ["kmag"],
    "CNN fusión espectrograma + k-space":    ["spec", "kmag"],
}
CNN_SCORES, CNN_PER_YEAR = {}, {}
for nm, kinds in tqdm(CNN_MODELS.items(), desc="CNN"):
    res_cnn, sc, vauc = kc.eval_cnn(d, INP, INP_CHAN, kinds, seeds=CNN_SEEDS, name=nm, epochs=CNN_EPOCHS,
                                    wandb_project=WANDB_PROJECT, wandb_group=WANDB_GROUP)
    CNN_SCORES[nm] = sc
    CNN_PER_YEAR[nm] = res_cnn[res_cnn.cond == "limpio"].set_index("hold").auc
    ROWS.append({"caracteristica": nm, "clasificador": "CNN", "dim": np.nan, **summarize(res_cnn), "domain acc": np.nan})
print("AUC por año evaluado (2022-24 = desarrollo; 2025 = prueba):"); display(pd.DataFrame(CNN_PER_YEAR).T.round(3))
board(rows=[r for r in ROWS if r["clasificador"] == "CNN"])

## 7. Tabla completa y gráfica

In [ ]:
full = board()
display(full)

top = full.head(12).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [1.3, 1]})
lab = top.caracteristica + "  |  " + top.clasificador
axes[0].barh(lab, top["AUC dev"], color="tab:blue", label="AUC dev (2022-24, criterio de elección)")
axes[0].scatter(top["AUC 2025"], lab, color="k", s=22, zorder=3, label="AUC 2025 (prueba)")
axes[0].scatter(top["AUC dev canal"], lab, facecolors="none", edgecolors="tab:red", s=22, zorder=3, label="AUC dev con canal sintético")
axes[0].set_xlim(0.6, None); axes[0].set_title("AUC Normal vs anormal (mejores 12 por AUC dev)"); axes[0].legend(loc="lower right", fontsize=7)
axes[0].tick_params(axis="y", labelsize=7)
ref = full[(full.caracteristica == "logK") & (full.clasificador == "LR+PCA64 (C=1)")].iloc[0]
axes[1].scatter(full["domain acc"], full["AUC dev"], c=np.where(full.caracteristica == "espectrograma", "tab:red", "tab:blue"))
axes[1].scatter([ref["domain acc"]], [ref["AUC dev"]], c="tab:orange", s=90, marker="*", label="punto de partida (logK, LR+PCA64)")
axes[1].axvline(1 / 3, color="k", ls="--", lw=1); axes[1].set_xlabel("domain acc (menor = más invariante)"); axes[1].set_ylabel("AUC dev")
axes[1].set_title("AUC dev vs invariancia al año"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 8. El mejor modelo: ROC por año y qué subclase falla

Se elige por `AUC dev` entre las filas de la tabla. ROC: líneas finas = folds de desarrollo (2022–24), línea gruesa = prueba en 2025.

In [ ]:
best = full[full.clasificador != "CNN"].iloc[0]
print("mejor no-CNN:", best.caracteristica, "|", best.clasificador, f"| AUC dev={best['AUC dev']:.3f} | AUC 2025={best['AUC 2025']:.3f}")
cnn_rows = full[full.clasificador == "CNN"]
best_cnn = cnn_rows.iloc[0] if len(cnn_rows) else None
if best_cnn is not None:
    print("mejor CNN   :", best_cnn.caracteristica, f"| AUC dev={best_cnn['AUC dev']:.3f} | AUC 2025={best_cnn['AUC 2025']:.3f}")

MAKERS = {**CLFS, "HistGradientBoosting": kl.clf_hgb}


def make_best():
    feat, cn = best.caracteristica, best.clasificador
    for nm, parts in COMBOS.items():
        if feat.rsplit(" [", 1)[0] == nm:
            F = np.hstack([FEATS[p] for p in parts]); dims = [FEATS[p].shape[1] for p in parts]
            if cn == "HistGradientBoosting":
                return F, kl.clf_hgb
            C = float(feat.split("C=")[1].rstrip("]"))
            return F, (lambda: kl.clf_lr(C, dims))
    mk = BEST_LIN if cn == best_lin_name else MAKERS[cn]
    return FEATS[feat], mk


Fb, mk = make_best()
scores_lin = np.full(len(d.y), np.nan)
for split, hold, tr, te in kl.protocol_folds(d):
    scores_lin[te] = mk().fit(Fb[tr], d.yb[tr]).predict_proba(Fb[te])[:, 1]

panels = [(f"{best.caracteristica}\n{best.clasificador}", scores_lin)]
if best_cnn is not None:
    panels.append((best_cnn.caracteristica, CNN_SCORES[best_cnn.caracteristica]))
fig, axes = plt.subplots(1, len(panels), figsize=(5.5 * len(panels), 5), squeeze=False)
for ax, (ttl, sc) in zip(axes[0], panels):
    for split, hold, tr, te in kl.protocol_folds(d):
        fpr, tpr, _ = roc_curve(d.yb[te], sc[te])
        tag = "prueba" if split == "test" else "dev"
        ax.plot(fpr, tpr, lw=2.5 if split == "test" else 1, label=f"{hold} ({tag})  AUC={roc_auc_score(d.yb[te], sc[te]):.3f}")
    ax.plot([0, 1], [0, 1], "k--", lw=0.8); ax.set_xlabel("FPR (normal clasificado como anormal)"); ax.set_ylabel("TPR")
    ax.set_title(ttl, fontsize=10); ax.legend()
plt.tight_layout(); plt.show()

sub_tab = full.head(8)[["caracteristica", "clasificador", "AUC dev", "AUC 2025", "AUC N-FC (2025)", "AUC N-W (2025)", "AUC N-WC (2025)"]]
sub_tab

## 9. Hallazgos de la corrida inicial

Corrida con `RANDOM_STATE=42`, `MAX_PER_YEAR_CLASS=1200`, ventana Tukey, 2 semillas en las CNN. `AUC dev` = media de los 3 folds de 2022–24; `AUC 2025` = entrenar con 2022–24 y evaluar en 2025. No hay intervalos de confianza: diferencias de ~0.02 pueden ser ruido (una sola muestra de datos, 2 semillas).

| Modelo | AUC dev | AUC 2025 |
|---|---|---|
| `logK` + LR/PCA64 (punto de partida) | 0.872 | 0.683 |
| espectrograma plano + LR/PCA64 | 0.750 | 0.624 |
| k-space local `local16 mean+max_t` + LR/PCA64 | **0.905** | **0.720** |
| CNN espectrograma | 0.972 | **0.760** |
| CNN espectrograma − media temporal | 0.975 | 0.741 |
| CNN imagen de k-space `log\|K\|` | 0.845 | 0.698 |
| CNN fusión espectrograma + k-space | 0.951 | 0.694 |

1. **Lo que sí mejoró (k-space).** El k-space **local** por parches 16×16 con `mean`+`max` y **posición en frecuencia** (`local16 mean+max_t`) sube 2025 de 0.683 a 0.720 y desarrollo de 0.872 a 0.905. Es lo mejor entre los modelos lineales en ambas columnas.
2. **El clasificador no es el cuello de botella (H1).** LR con distinto C, LDA y PCA 64/128 sobre `logK` quedan en 0.861–0.872 (dev) y 0.669–0.691 (2025).
3. **Quitar la posición en frecuencia (`_all`) engaña.** Rinde 0.80–0.87 en desarrollo pero **0.51–0.59 en 2025**: lo que aprende en 2022–24 no existe en 2025.
4. **`HistGradientBoosting` y combinar bloques (H5) no ayudan:** en 2025 los HGB sobre características compactas dan 0.46–0.61 y ninguna combinación supera a `local16 mean+max_t` solo (≤ 0.705).
5. **CNNs (H6).** Sobre espectrograma son la mejor en 2025 (0.74–0.76, por encima del mejor k-space lineal), pero la **imagen de k-space** no aporta (0.698, igual que el lineal sobre `logK`) y la **fusión empeora** (0.694). Restar la media temporal (notch `ku=0`) no cambia nada a la CNN sobre espectrograma.
6. **Dev no predice 2025.** Las CNN sobre espectrograma pasan de ≈ 0.97 en desarrollo a 0.74–0.76 en 2025 (la de k-space de 0.845 a 0.698, la fusión de 0.951 a 0.694), y el orden en desarrollo no coincide con el de 2025 (ver `_all`). Los folds de desarrollo comparten con el entrenamiento la misma augmentation (SpecAugment, ruido, pitch/stretch) que 2025 no tiene, así que no pueden revelar ese desajuste. Es una hipótesis, no está probada aquí.
7. **Robustez a canal.** Con el canal sintético los modelos en k-space casi no se mueven (≤ 0.01 de AUC; local16: 0.905 → 0.902) y el espectrograma plano cae de 0.750 a 0.668. Es esperable por construcción (`ku=0` cancela un canal fijo), no prueba robustez real.

**Siguiente paso con más información.** Regenerar `data/processed/train` con `stages.augmentation.enabled=false` (en `workflow/`, `python run.py … stages.augmentation.enabled=false`) y repetir: así desaparece el desajuste de augmentation entre desarrollo y prueba y se separa *shift real* de *artefactos de augmentation*. Las curvas por época y el AUC por fold de las CNN están en WandB, proyecto `biocas`, grupo `kspace-binary-cnn-v2`.